# Task 2 Demo - Cluster Interpretation and Submission Build

In this notebook I demonstrate how I validate cluster quality beyond a single score.

What I show:

1. Train one final clustering model.
2. Extract top terms per cluster for semantic interpretation.
3. Inspect representative documents per cluster.
4. Build and save `clusters.csv` in the required format and order.

Reader guide:

- This notebook is the final Task-2 interpretation and export step.
- It assumes a selected method + cluster count from earlier comparison.
- Every key artifact is saved to `data/results/`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import pairwise_distances

from core.data_io import ArticleDataset
from clustering import create_cluster_output
from preprocessing import TextNormalizer, TextPreprocessor

## Load data and create TF-IDF features

I use the same preprocessing configuration as in method comparison so interpretation stays consistent.

In [2]:
project_root_path = Path.cwd().parent
results_data_directory_path = project_root_path / "data" / "results"
results_data_directory_path.mkdir(parents=True, exist_ok=True)

articles_csv_path = project_root_path / "data" / "raw" / "articles.csv"
articles_data_frame = ArticleDataset(input_csv_path=articles_csv_path).load_articles()

text_normalizer = TextNormalizer()
normalized_bundle = text_normalizer.normalize_for_both_tasks(articles_data_frame["text"].tolist())

clustering_preprocessor = TextPreprocessor(
    vectorization_model_name="tfidf",
    max_features=20000,
    min_document_frequency=1,
    max_document_frequency=0.92,
    ngram_range=(1, 2),
    analyzer_mode="word",
)

clustering_feature_matrix = clustering_preprocessor.fit_transform(normalized_bundle.clustering_texts)
feature_names = np.array(clustering_preprocessor.get_feature_names())

clustering_feature_matrix.shape

(2164, 20000)

## Fit final model and record silhouette score

I keep this metric in a CSV so final model choice is documented with a quantitative value.

In [3]:
selected_cluster_count = 8

kmeans_model = KMeans(n_clusters=selected_cluster_count, n_init="auto", random_state=42)
cluster_labels = kmeans_model.fit_predict(clustering_feature_matrix)

silhouette_value = silhouette_score(clustering_feature_matrix, cluster_labels)
pd.DataFrame(
    [{"model": "kmeans", "cluster_count": selected_cluster_count, "silhouette_score": float(silhouette_value)}]
).to_csv(results_data_directory_path / "notebook_06_selected_model_metric.csv", index=False)

silhouette_value

0.012176493932852805

## Extract top terms per cluster

Top terms are taken from centroid weights in TF-IDF space.

This section supports qualitative interpretation in the report.

In [4]:
cluster_top_term_rows: list[dict[str, int | str | float]] = []

for cluster_label in range(selected_cluster_count):
    centroid_weights = kmeans_model.cluster_centers_[cluster_label]
    top_feature_indices = np.argsort(centroid_weights)[-15:][::-1]
    for term_rank, feature_index in enumerate(top_feature_indices, start=1):
        cluster_top_term_rows.append(
            {
                "cluster_label": cluster_label,
                "term_rank": term_rank,
                "term": str(feature_names[feature_index]),
                "weight": float(centroid_weights[feature_index]),
            }
        )

cluster_top_terms_data_frame = pd.DataFrame(cluster_top_term_rows)
cluster_top_terms_data_frame.to_csv(results_data_directory_path / "notebook_06_cluster_top_terms.csv", index=False)
cluster_top_terms_data_frame.head(20)

,cluster_label,term_rank,term,weight
0,0,1,intellect shameful,0.144182
1,0,2,shameful surrender,0.144182
2,0,3,shameful,0.144182
3,0,4,surrender soon,0.144182
4,0,5,skepticism chastity,0.143731
5,0,6,chastity,0.143731
6,0,7,chastity intellect,0.143731
7,0,8,banks skepticism,0.143731
8,0,9,skepticism,0.143285
9,0,10,surrender,0.143285


## Extract representative documents per cluster

I pick documents closest to each cluster centroid.

Why this helps:

- It shows concrete examples for each discovered topic.
- It supports manual semantic validation in the report.

In [5]:
representative_rows: list[dict[str, int | str | float]] = []

for cluster_label in range(selected_cluster_count):
    cluster_indices = np.where(cluster_labels == cluster_label)[0]
    if len(cluster_indices) == 0:
        continue

    centroid_vector = kmeans_model.cluster_centers_[cluster_label].reshape(1, -1)
    cluster_vectors_dense = clustering_feature_matrix[cluster_indices].toarray()
    centroid_distances = pairwise_distances(cluster_vectors_dense, centroid_vector, metric="cosine").ravel()

    nearest_positions = np.argsort(centroid_distances)[:5]
    for representative_rank, local_position in enumerate(nearest_positions, start=1):
        global_document_index = int(cluster_indices[local_position])
        representative_rows.append(
            {
                "cluster_label": cluster_label,
                "representative_rank": representative_rank,
                "doc_id": str(articles_data_frame.iloc[global_document_index]["doc_id"]),
                "distance_to_centroid": float(centroid_distances[local_position]),
                "snippet": str(articles_data_frame.iloc[global_document_index]["text"])[:220].replace("\n", " "),
            }
        )

representative_documents_data_frame = pd.DataFrame(representative_rows)
representative_documents_data_frame.to_csv(
    results_data_directory_path / "notebook_06_cluster_representative_docs.csv", index=False
)
representative_documents_data_frame.head(25)

,cluster_label,representative_rank,doc_id,distance_to_centroid,snippet
0,0,1,DOC_01195,0.094242,Senile keratoses. Have nothing to do with the ...
1,0,2,DOC_01690,0.112790,"By law, they would not be allowed to do that a..."
2,0,3,DOC_00512,0.135399,So just what was it you wanted to say? -- ----...
3,0,4,DOC_01280,0.160474,"""Diet Evangelist"". Good term. Fits Atkins to a..."
4,0,5,DOC_01819,0.214109,I'm not sure it is the fluctuation so much as ...
5,1,1,DOC_00810,0.667110,"*nnnnnnnng* Thank you for playing, I cannot ag..."
6,1,2,DOC_01877,0.705423,"ETHER IMPLODES 2 EARTH CORE, IS GRAVITY!!! Thi..."
7,1,3,DOC_01528,0.737753,"Howdy, I'm a little new to this newsgroup, but..."
8,1,4,DOC_01868,0.739428,... ... Some other owners on the ford-probe@wo...
9,1,5,DOC_01503,0.746839,"I'm not very impressed by the old so-called ""p..."


## Build submission-style clusters output

This block creates final cluster labels in assignment format and in original row order.

In [ ]:
cluster_output_data_frame = create_cluster_output(
    document_ids=articles_data_frame["doc_id"].tolist(),
    labels=cluster_labels,
)

# This keeps document order exactly aligned with the original dataset.
cluster_output_data_frame.to_csv(results_data_directory_path / "clusters.csv", index=False)
cluster_output_data_frame.to_csv(results_data_directory_path / "notebook_06_clusters_candidate.csv", index=False)
cluster_output_data_frame.head()

### Files produced by this notebook

- `data/results/notebook_06_selected_model_metric.csv`
- `data/results/notebook_06_cluster_top_terms.csv`
- `data/results/notebook_06_cluster_representative_docs.csv`
- `data/results/notebook_06_clusters_candidate.csv`
- `data/results/clusters.csv`

